In [1]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 22.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
     "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Mllama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 7.231 GiB
no_split classes   : ['MllamaCrossAttentionDecoderLayer', 'MllamaSelfAttentionDecoderLayer', 'MllamaVisionEncoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 9.188 GiB requested
  cuda:0  budget  12.95 GiB  weights  3.760 GiB  free  9.192 GiB  reserve  9.188 GiB
  cuda:1  budget  13.00 GiB  weights  3.471 GiB  free  9.525 Gi

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

In [4]:
ABDOMEN_ROOT = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT"
#NECK_ROOT  = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Neck"
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if root.endswith("Abdomen/CT") or root.endswith(os.path.join("Abdomen", "CT")):
        print(root)

/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT


In [5]:
# ---------------------------------------------------------------------------
# 4. Load your Abdomen dataset -- TRAIN split, open.csv / closed.csv
#    Pick one CSV to run first via WHICH, each capped to top N rows.
# ---------------------------------------------------------------------------
import os
import glob
import pandas as pd
from PIL import Image

#ABDOMEN_ROOT = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT"

WHICH = "open"      # change to "closed" for the second run
TOP_N = 150         # rows to take from each CSV

# Find train CSVs (open.csv / closed.csv) under the train split
train_csvs = glob.glob(os.path.join(ABDOMEN_ROOT, "train", "*.csv"))
print(f"Found {len(train_csvs)} training CSVs:")
for p in train_csvs:
    print(" ", p)

if not train_csvs:
    raise FileNotFoundError(
        f"No CSVs found under {ABDOMEN_ROOT}/*/train/*.csv -- check ABDOMEN_ROOT "
        f"and confirm your folder structure matches Abdomen/CT/train/*.csv"
    )

# Pick only the CSV matching WHICH (open.csv or closed.csv)
target_csv = None
for p in train_csvs:
    fname = os.path.basename(p).lower()
    if fname.startswith(WHICH):
        target_csv = p
        break

if target_csv is None:
    raise FileNotFoundError(f"No CSV matching '{WHICH}' found among: {train_csvs}")

split_dir = os.path.dirname(target_csv)   # e.g. Abdomen/CT/train
df = pd.read_csv(target_csv)
df["split_dir"] = split_dir

# Keep only the top N rows
df = df.head(TOP_N).reset_index(drop=True)

print(f"\nLoaded '{WHICH}' -> {len(df)} rows (capped at {TOP_N})")
print("Columns:", list(df.columns))

train_df = df  # this run's working dataframe

# Handle either column name -- "img_name" (original SLAKE export) or
# "image_file" (the real-image-files export from earlier).
IMG_COL = "image_file" if "image_file" in train_df.columns else "img_name"
print(f"Using image column: {IMG_COL}")

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(img_name)  # handles "xmlab1/source.jpg" -> "source.jpg"
    candidates = [
        os.path.join(split_dir, img_name),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, img_name.replace("/", "_")),  # matches earlier flatten step
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

Found 2 training CSVs:
  /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT/train/open.csv
  /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT/train/closed.csv

Loaded 'open' -> 150 rows (capped at 150)
Columns: ['image_file', 'img_id', 'location', 'modality', 'question', 'answer', 'q_lang', 'answer_type', 'content_type', 'base_type', 'qid', 'triple', 'split_dir']
Using image column: image_file


In [6]:
# ---------------------------------------------------------------------------
# 6. Your exact CoT instruction template
# ---------------------------------------------------------------------------
INSTRUCTION_TEMPLATE = """Context: You are a senior radiologist performing structured diagnostic reasoning.

Your goal is NOT just to answer, but to produce a **step-by-step clinical reasoning chain (Chain-of-Thought)** grounded in the image.

-----------------------------------
INPUT:
- Question: {question}
- Ground Truth Answer: {answer}
-----------------------------------

TASK INSTRUCTIONS:

You MUST follow a strict multi-step reasoning process:

Step 1: Identify Imaging Modality
- Determine modality (X-ray / CT / MRI / Ultrasound)
- Explain visual clues (contrast, density, grayscale pattern)

Step 2: Global Image Understanding
- Describe anatomical region
- Identify orientation (axial, sagittal, coronal, frontal)

Step 3: Region-wise Analysis
- Divide image into anatomical zones
- Analyze each region systematically

Step 4: Visual Feature Extraction
- Density (hyperdense / hypodense)
- Shape, edges, symmetry
- Texture abnormalities

Step 5: Abnormality Detection
- Identify pathology (if present)
- Localize precisely

Step 6: Clinical Reasoning
- Link findings to medical knowledge
- Explain WHY the abnormality matches the condition

Step 7: Question Understanding
- What exactly is the question asking?
- Type: (yes/no, location, modality, abnormality)

Step 8: Answer Justification
- Justify the provided answer: "{answer}"
- Explain why it is correct based on image evidence

-----------------------------------
OUTPUT FORMAT (STRICT):

1. Imaging Modality:
2. Anatomical Region:
3. Orientation:
4. Region-wise Findings:
5. Key Visual Features:
6. Detected Abnormality:
7. Clinical Interpretation:
8. Question Analysis:
9. Final Answer Justification:

IMPORTANT:
- Do NOT skip steps
- Do NOT give short answers
- Each step must contain 2-4 sentences
- Use medical terminology
"""

def generate_cot(image: Image.Image, question: str, answer: str) -> str:
    prompt_text = INSTRUCTION_TEMPLATE.format(question=question, answer=answer)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {"type": "image", "image": image},
            ],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,   # CoT is long: 9 sections x 2-4 sentences each
        use_cache=True,
        temperature=0.3,
        min_p=0.1,
    )
    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

In [7]:
# 7. Generate CoT for every row in train_df (in place, as a new column)
#    NOTE: This only touches train_df in memory + writes new output files.
#    Your original Drive zip / extracted Abdomen folder are never modified.
# ---------------------------------------------------------------------------
print(f"Generating CoT for {len(train_df)} training rows...")

cots = []
for i, row in train_df.iterrows():
    try:
        img_path = resolve_image_path(row["split_dir"], row[IMG_COL])
        image = Image.open(img_path).convert("RGB")
        cot = generate_cot(image, row["question"], row["answer"])
    except Exception as e:
        print(f"  [WARN] row {i} failed ({row.get(IMG_COL)}): {e}")
        cot = ""
    cots.append(cot)

    if i % 10 == 0:
        print(f"  [{i}/{len(train_df)}] done")

train_df["CoT"] = cots
print("\nCoT generation complete.")

Generating CoT for 150 training rows...


[unsloth_zoo.log|WARNING]Unsloth: torch.compile hit one of Unsloth's own `torch.compiler.disable`d gradient-checkpointing hooks inside MllamaPrecomputedAspectRatioEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed.
[unsloth_zoo.log|WARNING]Unsloth: torch.compile hit one of Unsloth's own `torch.compiler.disable`d gradient-checkpointing hooks inside MllamaPrecomputedPositionEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed.


  [0/150] done
  [10/150] done
  [20/150] done
  [30/150] done
  [40/150] done
  [50/150] done
  [60/150] done
  [70/150] done
  [80/150] done
  [90/150] done
  [100/150] done
  [110/150] done
  [120/150] done
  [130/150] done
  [140/150] done

CoT generation complete.


In [8]:
train_df.head()

,image_file,img_id,location,modality,question,answer,q_lang,answer_type,content_type,base_type,qid,triple,split_dir,CoT
0,xmlab104_source.jpg,104,Abdomen,CT,What modality is used to take this image?,CT,en,OPEN,Modality,vqa,22,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Identify Imaging Modality**\n\nThe i...
1,xmlab104_source.jpg,104,Abdomen,CT,Which part of the body does this image belong to?,Chest,en,OPEN,Position,vqa,23,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
2,xmlab104_source.jpg,104,Abdomen,CT,What is the main organ in the image?,Lung,en,OPEN,Organ,vqa,24,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
3,xmlab104_source.jpg,104,Abdomen,CT,What is the largest organ in the picture?,Lung,en,OPEN,Size,vqa,25,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
4,xmlab104_source.jpg,104,Abdomen,CT,What diseases are included in the picture?,Lung Cancer,en,OPEN,Abnormality,vqa,29,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...


In [9]:
# ---------------------------------------------------------------------------
# 8. Save back out to per-CSV files (grouped by original split_dir), into a
#    SEPARATE local output folder -- never overwriting your original files.
# ---------------------------------------------------------------------------
OUTPUT_ROOT = "/kaggle/working/Abdomen_Train_with_CoT"

for split_dir, group_df in train_df.groupby("split_dir"):
    rel_dir = os.path.relpath(split_dir, ABDOMEN_ROOT)   # e.g. "CT/train"
    out_dir = os.path.join(OUTPUT_ROOT, rel_dir)
    os.makedirs(out_dir, exist_ok=True)

    # Figure out which original csv (open/closed) each row came from, by
    # matching answer_type back to the filename convention used earlier.
    for answer_type, at_df in group_df.groupby("answer_type"):
        out_name = f"{answer_type.lower()}.csv"   # open.csv / closed.csv
        out_path = os.path.join(out_dir, out_name)
        at_df.drop(columns=["split_dir"]).to_csv(out_path, index=False)
        print(f"Saved: {out_path}  ({len(at_df)} rows)")

print(f"\nAll done. CoT-augmented train CSV saved under: {OUTPUT_ROOT}")
print("Your original zip and extracted folder were never modified.")

Saved: /kaggle/working/Abdomen_Train_with_CoT/train/open.csv  (150 rows)

All done. CoT-augmented train CSV saved under: /kaggle/working/Abdomen_Train_with_CoT
Your original zip and extracted folder were never modified.


In [10]:
# ---------------------------------------------------------------------------
# 8. Save back out to per-CSV files (grouped by original split_dir), into a
#    SEPARATE local output folder -- never overwriting your original files.
# ---------------------------------------------------------------------------
OUTPUT_ROOT = f"/kaggle/working/Abdomen_Train_with_CoT_{WHICH}"

for split_dir, group_df in train_df.groupby("split_dir"):
    rel_dir = os.path.relpath(split_dir, ABDOMEN_ROOT)   # e.g. "CT/train"
    out_dir = os.path.join(OUTPUT_ROOT, rel_dir)
    os.makedirs(out_dir, exist_ok=True)

    for answer_type, at_df in group_df.groupby("answer_type"):
        out_name = f"{answer_type.lower()}.csv"   # open.csv / closed.csv
        out_path = os.path.join(out_dir, out_name)
        at_df.drop(columns=["split_dir"]).to_csv(out_path, index=False)
        print(f"Saved: {out_path}  ({len(at_df)} rows)")

print(f"\nAll done. CoT-augmented '{WHICH}' train CSV saved under: {OUTPUT_ROOT}")
print("Your original zip and extracted folder were never modified.")

Saved: /kaggle/working/Abdomen_Train_with_CoT_open/train/open.csv  (150 rows)

All done. CoT-augmented 'open' train CSV saved under: /kaggle/working/Abdomen_Train_with_CoT_open
Your original zip and extracted folder were never modified.


In [11]:
import shutil
import os

OUTPUT_ROOT = f"/kaggle/working/Abdomen_Train_with_CoT_{WHICH}"
ZIP_PATH = f"/kaggle/working/Abdomen_Train_with_CoT_{WHICH}_V2"  # shutil appends .zip automatically

shutil.make_archive(ZIP_PATH, "zip", OUTPUT_ROOT)

zip_file = ZIP_PATH + ".zip"
print(f"Zipped to: {zip_file}")
print(f"Size: {os.path.getsize(zip_file) / (1024*1024):.2f} MB")

Zipped to: /kaggle/working/Abdomen_Train_with_CoT_open_V2.zip
Size: 0.06 MB
